# IMP Demo — Collaborator Guide

Interactive sandbox over `src/`. Use this notebook to try **new data, features, and models** before promoting anything to `main.py`.

## Prerequisites

- Put required parquets under `data/` (see `config/default.py`). Yahoo macros (`MACRO_FEATURES`) download/cache on first aligned-panel build.
- Run cells **top to bottom**. After changing CONFIG or §3 / §5+, re-run from that section through §7.
- If you edit `src/`, restart the kernel (or re-run the import cell) before fitting again.

## Where to edit

| Goal | Edit here | Notes |
| ---- | --------- | ----- |
| Target / split / benchmark / families | **§0a** `TARGET`, `BENCHMARK_SOURCE`, `FEATURE_COLUMNS`, `FAMILIES`, … | Comment/uncomment feature names to A/B. Benchmark: `yahoo` / `equal` / `cap`. |
| Built-in lin / tr / nn knobs | **§0b / 0c / 0d** | Only families listed in `FAMILIES` actually run. |
| New daily table (e.g. macro) | **§3** — merge onto `df_align` by `Date` | Market-level: `Date` only. Stock-level: `Date` + `TickerId`. |
| Features `main.py` already builds | **§0a** `FEATURE_COLUMNS` | QScore, analyst counts, derived ratings, macros, and engineered columns are on `df_align` after §2. Uncomment a name to fit it. |
| Event / preprocess / metrics | **§4–§7** | Prefer not to change wiring unless you know the pipeline. |
| Custom model (demo-only) | **§5+a / 5+b / 5+c** | Enable `"custom"` in `FAMILIES`. Leave **Wire-up** alone. |

## Feature checklist

1. The `main.py` feature menu is already built on `df_align` in **§2**. Uncomment names in **§0a** `FEATURE_COLUMNS`.
2. A column that is not on that menu is merged in **§3** (DecisionDate-safe; no lookahead), then added to `FEATURE_COLUMNS`.
3. Re-run **§4 → §7** (not only fit).

## Custom model checklist

1. Uncomment `"custom"` in **§0a** `FAMILIES` (alone or with `lin` / `tr` / `nn`).
2. Edit only **§5+a / 5+b / 5+c** (hyperparams + fitter). Roles mirror lin / tr / nn.
3. Re-run **§5+ → §7**. Do not edit the Wire-up cell unless adding a new model id.

## Demo vs `main.py`

- §3 extra merges and `"custom"` stay in this notebook unless you later move them into `src/` / `config/` / `model/`.
- Active `FEATURE_COLUMNS` match `config/default.py`: `QScoreTrendLong`, `QScoreSpeedLong`, `mom_21d`, `vol_21d`, `log_mcap`. The other main-menu names are commented in §0a.
- Batch baseline: `python main.py` with the same `FAMILIES` idea for `lin` / `tr` / `nn` only.


## 0. Config

Shared knobs (including which families to run), then hyperparameter cells for
`LIN` / `TR` / `NN`. Those family cells always define settings; only `FAMILIES`
in **0a** decides what actually runs.


In [36]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import src.train as _train
importlib.reload(_train)

from src.report import print_notebook_config
from src.train import resolve_demo_setup


In [37]:
pd.set_option("display.width", None) 
pd.set_option("display.expand_frame_repr", False)  
pd.set_option("display.max_columns", None)

### 0a. Shared


In [38]:
# Shared across families (target / split / benchmark / importance).
# Notebook-local knobs — pass these explicitly below; do not rely on config.default.
TARGET = "ret_ann_mkt"
# Benchmark for *_mkt labels: "yahoo" | "equal" | "cap"
BENCHMARK_SOURCE = "equal"
# Used only when BENCHMARK_SOURCE == "yahoo".
MARKET_TICKER = "SPY"
# Same menu as config/default.py FEATURE_COLUMNS. Uncomment a name to fit it.
# Original 8, derived ratings, macros, and engineered columns are attached before §4.
FEATURE_COLUMNS = [
    "QScoreTrendLong",
    "QScoreSpeedLong",
    # "QScoreLong",
    # "analystRatingsStrongBuy",
    # "analystRatingsBuy",
    # "analystRatingsHold",
    # "analystRatingsSell",
    # "analystRatingsStrongSell",
    # "Coverage",
    # "BuyRatio",
    # "SellRatio",
    # "RatingScore",
    # "vix",
    # "tnx",
    # "dxy",
    "mom_21d",
    "vol_21d",
    "log_mcap",
    # "par_mkt",
    # "par*vix",
]

SPLIT_COLUMN = "DecisionDate"
TRAIN_END = "2022-12-31"
VAL_END = "2023-12-31"
IMPUTE_STRATEGY = "median"
RANDOM_STATE = 42
FEATURE_IMPORTANCE = False
FEATURE_IMPORTANCE_REPEATS = 10

# Which families to run (same idea as main.py FAMILIES).
# "custom" is demo-only; define CUSTOM + fit_custom in §5+.
FAMILIES = (
    "lin",
    # "tr",
    # "nn",
    # "custom",
)


### 0b. Linear

Hyperparameters for the linear family (used when `"lin"` is in `FAMILIES`).
Defaults match `config/config_lin.py`.


In [39]:
LIN = {
    "MODEL_TYPES": ("ols", "ridge", "lasso"),
    "ALPHA": 1.0,
    "ALPHA_GRID": (0.01, 0.1, 1.0, 10.0, 100.0),
    "STANDARDIZE": True,
}


### 0c. Trees

Hyperparameters for the tree family (used when `"tr"` is in `FAMILIES`).
Defaults match `config/config_tr.py`.


In [40]:
TR = {
    "MODEL_TYPES": (
        "decision_tree",
        "random_forest",
        "hist_gradient_boosting",
    ),
    "N_ESTIMATORS": 200,
    "MAX_DEPTH": 4,
    "MAX_DEPTH_GRID": (2, 3, 4, 6, 8),
    "LEARNING_RATE": 0.05,
    "MIN_SAMPLES_LEAF": 20,
    "EARLY_STOPPING_PATIENCE": 10,
    "STANDARDIZE": False,
}


### 0d. Neural

Hyperparameters for the neural family (used when `"nn"` is in `FAMILIES`).
Defaults match `config/config_nn.py`.


In [41]:
NN = {
    "MODEL_TYPES": ("mlp_shallow", "mlp", "mlp_deep"),
    "HIDDEN_SIZES": {
        "mlp_shallow": (64,),
        "mlp": (64, 32),
        "mlp_deep": (128, 64, 32),
    },
    "ALPHA": 1e-4,
    "ALPHA_GRID": (1e-5, 1e-4, 1e-3, 1e-2),
    "LR": 1e-3,
    "BATCH_SIZE": 64,
    "MAX_ITER": 200,
    "EARLY_STOPPING_PATIENCE": 8,
    "STANDARDIZE": True,
}


### Resolve Active Families And Data Paths


In [42]:
setup = resolve_demo_setup(FAMILIES, lin=LIN, tr=TR, nn=NN, root=ROOT)
ACTIVE_FAMILIES = list(setup.active_families)
STANDARDIZE = setup.standardize
DATA_DIR = setup.data_dir


In [43]:
print_notebook_config(
    target=TARGET,
    feature_columns=FEATURE_COLUMNS,
    split_column=SPLIT_COLUMN,
    train_end=TRAIN_END,
    val_end=VAL_END,
    impute_strategy=IMPUTE_STRATEGY,
    random_state=RANDOM_STATE,
    feature_importance=FEATURE_IMPORTANCE,
    feature_importance_repeats=FEATURE_IMPORTANCE_REPEATS,
    standardize=STANDARDIZE,
    active_families=ACTIVE_FAMILIES,
    benchmark_source=BENCHMARK_SOURCE,
    market_ticker=MARKET_TICKER,
)


------------------------------------------------------------
CONFIG
------------------------------------------------------------
ACTIVE_FAMILIES  ['lin']
TARGET            ret_ann_mkt
BENCHMARK_SOURCE  equal
MARKET_TICKER     SPY
FEATURE_COLUMNS   (5)
                  - QScoreTrendLong
                  - QScoreSpeedLong
                  - mom_21d
                  - vol_21d
                  - log_mcap
SPLIT_COLUMN      DecisionDate
TRAIN_END         2022-12-31
VAL_END           2023-12-31
IMPUTE_STRATEGY   median
STANDARDIZE (step5 preview) True
RANDOM_STATE      42
FEATURE_IMPORTANCE False
FEATURE_IMPORTANCE_REPEATS 10

[lin]
  MODEL_TYPES            ('ols', 'ridge', 'lasso')
  ALPHA                  1.0
  ALPHA_GRID             (0.01, 0.1, 1.0, 10.0, 100.0)
  STANDARDIZE            True


## 1. Load And Merge

Load the three Parquet tables and build the daily panel `df_all` via `src.prepare` (`load_stock`, `load_earnings`, `load_sentiment`, `merge_panel`).

In [44]:
from src.prepare import load_earnings, load_sentiment, load_stock, merge_panel

In [45]:
stock_data = load_stock(DATA_DIR)
stock_data.head()

          | 0/1376761

,TickerId,ShortName,InstrumentName,Date,LastPrice,High,Open,Low,Volume,CurrencyId,QScoreTrendLong,QScoreSpeedLong,QScoreLong,MarketCap
0,1229,A:XNYS,"Agilent Technologies, Inc.",2025-12-31,136.0700,137.9700,137.6100,136.0400,949976.0000,840,-5.0000,-1.0000,-6.0000,38575691000.0000
1,1229,A:XNYS,"Agilent Technologies, Inc.",2025-12-30,137.6200,138.1600,137.5700,136.8301,2423111.0000,840,-5.0000,-0.8571,-5.8571,39015115000.0000
2,1229,A:XNYS,"Agilent Technologies, Inc.",2025-12-29,137.9300,139.0700,138.4000,137.6200,1640851.0000,840,-5.0000,-1.4286,-6.4286,39102999000.0000
3,1229,A:XNYS,"Agilent Technologies, Inc.",2025-12-26,138.3900,138.6300,138.5000,137.8700,739337.0000,840,-5.0000,-1.8571,-6.8571,39233409000.0000
4,1229,A:XNYS,"Agilent Technologies, Inc.",2025-12-24,138.3200,138.5700,138.3500,137.7700,508961.0000,840,-5.0000,-2.7143,-7.7143,39213564000.0000


In [46]:
earnings = load_earnings(DATA_DIR)
earnings.head()

          | 0/20365

,Id,TickerId,EventDate,MarketTimeId
126,672683,2755,2023-08-31,1.0000
214,673904,13375,2019-01-28,2.0000
215,673905,13375,2019-04-29,2.0000
216,673906,13375,2019-07-29,2.0000
217,673907,13375,2019-10-28,2.0000


In [47]:
sentiment = load_sentiment(DATA_DIR)
sentiment.head()

          | 0/15471

,symbol,date,analystRatingsStrongBuy,analystRatingsBuy,analystRatingsHold,analystRatingsSell,analystRatingsStrongSell,ShortName_before,tickerId
3827,DAL,2019-01-01,7,11,3,0,0,DAL,15635
12862,TDY,2019-01-01,1,3,3,0,0,TDY,4219
5239,EXR,2019-01-01,3,5,10,1,0,EXR,2308
6528,GPN,2019-01-01,10,16,9,1,0,GPN,2564
7648,INVH,2019-01-01,6,9,2,0,0,INVH,11758


In [48]:
df_all = merge_panel(stock_data, earnings, sentiment)
df_all

          | 0/1376761

,TickerId,ShortName,InstrumentName,Date,LastPrice,High,Open,Low,Volume,CurrencyId,QScoreTrendLong,QScoreSpeedLong,QScoreLong,MarketCap,Id,EventDate,MarketTimeId,Symbol,Exchange,symbol,date,analystRatingsStrongBuy,analystRatingsBuy,analystRatingsHold,analystRatingsSell,analystRatingsStrongSell,ShortName_before,tickerId
0,1229,A:XNYS,"Agilent Technologies, Inc.",2025-12-31,136.0700,137.9700,137.6100,136.0400,949976.0000,840,-5.0000,-1.0000,-6.0000,38575691000.0000,NaN,NaT,NaN,A,XNYS,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1229,A:XNYS,"Agilent Technologies, Inc.",2025-12-30,137.6200,138.1600,137.5700,136.8301,2423111.0000,840,-5.0000,-0.8571,-5.8571,39015115000.0000,NaN,NaT,NaN,A,XNYS,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1229,A:XNYS,"Agilent Technologies, Inc.",2025-12-29,137.9300,139.0700,138.4000,137.6200,1640851.0000,840,-5.0000,-1.4286,-6.4286,39102999000.0000,NaN,NaT,NaN,A,XNYS,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1229,A:XNYS,"Agilent Technologies, Inc.",2025-12-26,138.3900,138.6300,138.5000,137.8700,739337.0000,840,-5.0000,-1.8571,-6.8571,39233409000.0000,NaN,NaT,NaN,A,XNYS,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1229,A:XNYS,"Agilent Technologies, Inc.",2025-12-24,138.3200,138.5700,138.3500,137.7700,508961.0000,840,-5.0000,-2.7143,-7.7143,39213564000.0000,NaN,NaT,NaN,A,XNYS,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1376756,19311,EMB:XNAS,iShares J.P. Morgan USD Emerging Markets Bond ETF,2019-01-08,72.9942,73.0150,72.8351,72.6831,4462193.0000,840,3.0000,5.0000,8.0000,11289981172.0797,NaN,NaT,NaN,EMB,XNAS,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1376757,19311,EMB:XNAS,iShares J.P. Morgan USD Emerging Markets Bond ETF,2019-01-07,73.0495,73.1187,72.8904,72.8074,8315168.0000,840,2.2857,4.8571,7.1429,11298541483.8394,NaN,NaT,NaN,EMB,XNAS,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1376758,19311,EMB:XNAS,iShares J.P. Morgan USD Emerging Markets Bond ETF,2019-01-04,72.7382,72.8351,72.3370,72.3370,6787802.0000,840,1.5714,4.1429,5.7143,11250389730.1911,NaN,NaT,NaN,EMB,XNAS,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1376759,19311,EMB:XNAS,iShares J.P. Morgan USD Emerging Markets Bond ETF,2019-01-03,72.1917,72.4407,71.8873,71.8389,3988014.0000,840,0.8571,3.2857,4.1429,11165856651.5640,NaN,NaT,NaN,EMB,XNAS,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
df_summary = df_all.describe(include="all").T
df_summary["dtype"] = df_all.dtypes
df_summary

,count,unique,top,freq,mean,min,25%,50%,75%,max,std,dtype
TickerId,1376761.0000,NaN,NaN,NaN,7279.2042,1229.0000,2698.0000,4242.0000,12529.0000,19311.0000,5347.1381,int64
ShortName,1376761,815,BLK:XNYS,3520,NaN,NaN,NaN,NaN,NaN,NaN,NaN,string
InstrumentName,1376761,805,Brown-Forman Corporation,3520,NaN,NaN,NaN,NaN,NaN,NaN,NaN,string
Date,1376761,NaN,NaN,NaN,2022-07-24 19:15:28.613462016,2019-01-02 00:00:00,2020-10-30 00:00:00,2022-08-03 00:00:00,2024-04-24 00:00:00,2025-12-31 00:00:00,NaN,datetime64[ns]
LastPrice,1376761.0000,NaN,NaN,NaN,135.7771,0.2199,42.5399,80.6266,152.9180,9924.4000,266.9979,float64
High,1376761.0000,NaN,NaN,NaN,137.4221,0.2579,43.1000,81.5895,154.8174,9964.7700,270.1271,float64
Open,1376761.0000,NaN,NaN,NaN,135.7618,0.2326,42.5333,80.6100,152.9000,9914.1700,266.9694,float64
Low,1376761.0000,NaN,NaN,NaN,134.0754,0.2092,41.9612,79.6112,150.9726,9794.0000,263.8757,float64
Volume,1376761.0000,NaN,NaN,NaN,5864040.5668,2.0000,780584.0000,1862052.0000,4772826.0000,2511526760.0000,20745944.0803,float64
CurrencyId,1376761.0000,NaN,NaN,NaN,840.0000,840.0000,840.0000,840.0000,840.0000,840.0000,0.0000,int64


## 2. Aligned Panel And Benchmark

Build `df_align` with `src.events.build_aligned_panel` (uses `src.sessions` + `src.benchmark`). Pass §0a `BENCHMARK_SOURCE` / `MARKET_TICKER` explicitly so the demo does not inherit `config.default`.


In [50]:
from src.events import build_aligned_panel, build_event_table
from src.report import print_event_report


In [51]:
df_align = build_aligned_panel(
    df_all,
    data_dir=DATA_DIR,
    market_ticker=MARKET_TICKER,
    benchmark_source=BENCHMARK_SOURCE,
)
df_align.head()

          | 0/1301701

          | 0/1759

          | 0/1762

,TickerId,ShortName,InstrumentName,Date,LastPrice,High,Open,Low,Volume,CurrencyId,QScoreTrendLong,QScoreSpeedLong,QScoreLong,MarketCap,Id,EventDate,MarketTimeId,Symbol,Exchange,analystRatingsStrongBuy,analystRatingsBuy,analystRatingsHold,analystRatingsSell,analystRatingsStrongSell,BenchmarkAdjClose,vix,tnx,dxy,Coverage,BuyRatio,SellRatio,RatingScore,mom_21d,vol_21d,log_mcap,par_mkt,par*vix
0,1229,A:XNYS,"Agilent Technologies, Inc.",2019-01-02,62.8115,63.6529,63.5860,62.4386,2113304.0000,840,-2.7143,-1.4286,-4.1429,18573479868.3392,NaN,NaT,NaN,A,XNYS,NaN,NaN,NaN,NaN,NaN,NaN,23.2200,2.6610,96.8200,NaN,NaN,NaN,NaN,NaN,NaN,23.6450,NaN,NaN
1,1229,A:XNYS,"Agilent Technologies, Inc.",2019-01-03,60.4975,62.8976,62.6585,59.2832,5383926.0000,840,-2.4286,-1.2857,-3.7143,17889238411.7799,NaN,NaT,NaN,A,XNYS,NaN,NaN,NaN,NaN,NaN,97.9871,25.4500,2.5540,96.3100,NaN,NaN,NaN,NaN,NaN,NaN,23.6075,NaN,NaN
2,1229,A:XNYS,"Agilent Technologies, Inc.",2019-01-04,62.5916,63.0601,61.2816,61.2816,3123654.0000,840,-1.7143,-0.4286,-2.1429,18508448655.5257,NaN,NaT,NaN,A,XNYS,NaN,NaN,NaN,NaN,NaN,101.3373,21.3800,2.6590,96.1900,NaN,NaN,NaN,NaN,NaN,NaN,23.6415,NaN,NaN
3,1229,A:XNYS,"Agilent Technologies, Inc.",2019-01-07,63.9207,64.4705,62.7637,62.7350,3235055.0000,840,-1.0000,0.0000,-1.0000,18901463376.4420,NaN,NaT,NaN,A,XNYS,NaN,NaN,NaN,NaN,NaN,102.5425,21.4000,2.6820,95.7100,NaN,NaN,NaN,NaN,NaN,NaN,23.6625,NaN,NaN
4,1229,A:XNYS,"Agilent Technologies, Inc.",2019-01-08,64.8577,65.2211,64.6282,63.7773,1578055.0000,840,-0.2857,0.2857,0.0000,19178552891.9082,NaN,NaT,NaN,A,XNYS,NaN,NaN,NaN,NaN,NaN,103.7129,20.4700,2.7160,95.9000,NaN,NaN,NaN,NaN,NaN,NaN,23.6771,NaN,NaN


## 3. New Customized Data / Features

Yahoo macros (`vix` / `tnx` / `dxy`) are already attached in §2 via `config.default.MACRO_FEATURES` (cached like the yahoo benchmark under `data/macro_daily.*`). To use them in models, uncomment their names in §0a `FEATURE_COLUMNS`.

Optional: merge any *additional* daily table onto `df_align` by `Date` below.

In [52]:
# Macros are on df_align already (from MACRO_FEATURES). Preview:
df_align[["Date", "vix", "tnx", "dxy"]].drop_duplicates("Date").head()

,Date,vix,tnx,dxy
0,2019-01-02,23.2200,2.6610,96.8200
1,2019-01-03,25.4500,2.5540,96.3100
2,2019-01-04,21.3800,2.6590,96.1900
3,2019-01-07,21.4000,2.6820,95.7100
4,2019-01-08,20.4700,2.7160,95.9000


In [53]:
# Example: merge an extra local table (not required for vix/tnx/dxy).
""" extra = pd.read_parquet(DATA_DIR / "my_extra_daily.parquet")
df_align = df_align.merge(extra, on="Date", how="left")
df_align.head() """

' extra = pd.read_parquet(DATA_DIR / "my_extra_daily.parquet")\ndf_align = df_align.merge(extra, on="Date", how="left")\ndf_align.head() '

Build decision-day features on `df_align`: `mom_21d`, `vol_21d`, `log_mcap` (no lookahead; known at that close).

In [54]:
""" WINDOW = 21

df_align = df_align.sort_values(["TickerId", "Date"]).reset_index(drop=True)
price = df_align.groupby("TickerId", sort=False)["LastPrice"]
ret_1d = price.pct_change()
mcap = pd.to_numeric(df_align["MarketCap"], errors="coerce")

# Calculate the percentage change over the last 21 days 
df_align["mom_21d"] = price.pct_change(WINDOW)

# Calculate the standard deviation of the daily percentage changes over the last 21 days
df_align["vol_21d"] = (
    ret_1d.groupby(df_align["TickerId"], sort=False)
    .rolling(WINDOW, min_periods=WINDOW)
    .std(ddof=1)
    .reset_index(level=0, drop=True)
)

# Calculate the log of market cap
df_align["log_mcap"] = np.where(mcap > 0, np.log(mcap), np.nan)

df_align.head() """

' WINDOW = 21\n\ndf_align = df_align.sort_values(["TickerId", "Date"]).reset_index(drop=True)\nprice = df_align.groupby("TickerId", sort=False)["LastPrice"]\nret_1d = price.pct_change()\nmcap = pd.to_numeric(df_align["MarketCap"], errors="coerce")\n\n# Calculate the percentage change over the last 21 days \ndf_align["mom_21d"] = price.pct_change(WINDOW)\n\n# Calculate the standard deviation of the daily percentage changes over the last 21 days\ndf_align["vol_21d"] = (\n    ret_1d.groupby(df_align["TickerId"], sort=False)\n    .rolling(WINDOW, min_periods=WINDOW)\n    .std(ddof=1)\n    .reset_index(level=0, drop=True)\n)\n\n# Calculate the log of market cap\ndf_align["log_mcap"] = np.where(mcap > 0, np.log(mcap), np.nan)\ndf_align.head() '

## 4. Event Table

Build one-row-per-announcement `df` with `src.events.build_event_table` (reuses `df_align`); print descriptive stats with `src.report.print_event_report`.


In [55]:
df = build_event_table(
    df_all,
    data_dir=DATA_DIR,
    df_align=df_align,
    feature_columns=FEATURE_COLUMNS,
    market_ticker=MARKET_TICKER,
    benchmark_source=BENCHMARK_SOURCE,
)
print_event_report(df, primary_target=TARGET, feature_columns=FEATURE_COLUMNS)


          | 0/19697


------------------------------------------------------------
df coverage
------------------------------------------------------------
rows=19495  cols=23  tickers=748
EventDate        2019-01-03 .. 2025-12-19
DecisionDate     2019-01-02 .. 2025-12-17
MarketTimeId
1    10630
2     8865

------------------------------------------------------------
df head
------------------------------------------------------------
   TickerId Symbol ShortName   EventDate ReleaseDate  AnchorDate DecisionDate  MarketTimeId  ret_ann  ret_ann_mkt  ret_next  ret_next_mkt  ret_post  ret_post_mkt  dir_ann  vol_ann  vol_post  QScoreTrendLong  QScoreSpeedLong  mom_21d  vol_21d  log_mcap      Id
0      1229      A    A:XNYS  2019-02-20  2019-02-20  2019-02-21   2019-02-19             2   0.0116       0.0037   -0.0085       -0.0054    0.0199        0.0143        1   0.0113    0.0090           5.0000           2.1429   0.0946   0.0099   23.8106  726103
1      1229      A    A:XNYS  2019-05-14  2019-05-14  2019-05-

## 5. Preprocess (Train / Val / Test)

Chronological split and train-only impute/scale via `src.preprocess.run_preprocess`; report with `src.report.print_preprocess_report`.


In [56]:
from src.preprocess import run_preprocess
from src.report import print_preprocess_report

In [57]:
result = run_preprocess(
    df,
    target=TARGET,
    feature_columns=FEATURE_COLUMNS,
    split_column=SPLIT_COLUMN,
    train_end=TRAIN_END,
    val_end=VAL_END,
    impute_strategy=IMPUTE_STRATEGY,
    standardize=STANDARDIZE,
)
print_preprocess_report(result, target=TARGET, split_column=SPLIT_COLUMN)
train_y_mean = float(result.folds["train"].y.to_numpy(dtype=float).mean())



------------------------------------------------------------
folds
------------------------------------------------------------
train n=10736  2019-01-07 .. 2022-12-20
val   n= 2861  2023-01-03 .. 2023-12-20
test  n= 5883  2024-01-02 .. 2025-12-17
train_y_mean 0.0032095893946861367

------------------------------------------------------------
target summary
------------------------------------------------------------
target=ret_ann_mkt  events_in=19495  dropped_target=15
       count  unique     min    max   mean    std  missing_n missing_rate
fold                                                                     
train  10736   10675 -0.9504 0.5408 0.0032 0.0713          0        0.00%
val     2861    2840 -0.4579 0.5849 0.0049 0.0750          0        0.00%
test    5883    5843 -0.4653 0.7986 0.0034 0.0843          0        0.00%

------------------------------------------------------------
train X head
------------------------------------------------------------
   QScoreTrendLon

## 5+. Custom Model (demo-only)

Enable with `"custom"` in `FAMILIES` (§0a). Collaborator blocks map to main-family roles:

| Block | Role (like) | Model id |
| ----- | ----------- | -------- |
| **5+a** | `lin` | `elasticnet` |
| **5+b** | `tr` | `extratrees` |
| **5+c** | `nn` | `mlp_linear` |

Edit only 5+a/b/c. **Wire-up** is infrastructure.


In [58]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import ElasticNet
from sklearn.neural_network import MLPRegressor

from src.preprocess import PreprocessResult
from src.train import FitBundle

# Which custom models to run (comment a line to skip that lin/tr/nn slot).
CUSTOM_MODEL_TYPES = (
    # "elasticnet",   
    # "extratrees",   
    # "mlp_linear",   
)
CUSTOM_STANDARDIZE = True

# Filled by 5+a / 5+b / 5+c; do not edit by hand.
_CUSTOM_FITTERS = {}
_CUSTOM_PARAMS = {}


### 5+a. `lin` slot → `elasticnet`

Custom stand-in for the **linear** family. Edit only this block.


In [59]:
""" # --- collaborator edit: lin / elasticnet ---
ELASTICNET = {
    "ALPHA": 1.0,
    "L1_RATIO": 0.5,
}


def fit_elasticnet(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    alpha = float(ELASTICNET["ALPHA"])
    l1_ratio = float(ELASTICNET["L1_RATIO"])
    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=10_000,
        random_state=RANDOM_STATE,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={"alpha": alpha, "l1_ratio": l1_ratio},
    )


_CUSTOM_FITTERS["elasticnet"] = fit_elasticnet
_CUSTOM_PARAMS.update(ELASTICNET) """


' # --- collaborator edit: lin / elasticnet ---\nELASTICNET = {\n    "ALPHA": 1.0,\n    "L1_RATIO": 0.5,\n}\n\n\ndef fit_elasticnet(result: PreprocessResult) -> FitBundle:\n    train = result.folds["train"]\n    alpha = float(ELASTICNET["ALPHA"])\n    l1_ratio = float(ELASTICNET["L1_RATIO"])\n    model = ElasticNet(\n        alpha=alpha,\n        l1_ratio=l1_ratio,\n        max_iter=10_000,\n        random_state=RANDOM_STATE,\n    )\n    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))\n    return FitBundle(\n        model=model,\n        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),\n        summary_extras={"alpha": alpha, "l1_ratio": l1_ratio},\n    )\n\n\n_CUSTOM_FITTERS["elasticnet"] = fit_elasticnet\n_CUSTOM_PARAMS.update(ELASTICNET) '

### 5+b. `tr` slot → `extratrees`

Custom stand-in for the **tree** family. Edit only this block.


In [60]:
""" # --- collaborator edit: tr / extratrees ---
EXTRATREES = {
    "N_ESTIMATORS": 200,
    "MAX_DEPTH": 6,
    "MIN_SAMPLES_LEAF": 20,
}


def fit_extratrees(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    model = ExtraTreesRegressor(
        n_estimators=int(EXTRATREES["N_ESTIMATORS"]),
        max_depth=int(EXTRATREES["MAX_DEPTH"]),
        min_samples_leaf=int(EXTRATREES["MIN_SAMPLES_LEAF"]),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={
            "n_estimators": int(EXTRATREES["N_ESTIMATORS"]),
            "max_depth": int(EXTRATREES["MAX_DEPTH"]),
        },
    )


_CUSTOM_FITTERS["extratrees"] = fit_extratrees
_CUSTOM_PARAMS.update(EXTRATREES) """


' # --- collaborator edit: tr / extratrees ---\nEXTRATREES = {\n    "N_ESTIMATORS": 200,\n    "MAX_DEPTH": 6,\n    "MIN_SAMPLES_LEAF": 20,\n}\n\n\ndef fit_extratrees(result: PreprocessResult) -> FitBundle:\n    train = result.folds["train"]\n    model = ExtraTreesRegressor(\n        n_estimators=int(EXTRATREES["N_ESTIMATORS"]),\n        max_depth=int(EXTRATREES["MAX_DEPTH"]),\n        min_samples_leaf=int(EXTRATREES["MIN_SAMPLES_LEAF"]),\n        random_state=RANDOM_STATE,\n        n_jobs=-1,\n    )\n    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))\n    return FitBundle(\n        model=model,\n        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),\n        summary_extras={\n            "n_estimators": int(EXTRATREES["N_ESTIMATORS"]),\n            "max_depth": int(EXTRATREES["MAX_DEPTH"]),\n        },\n    )\n\n\n_CUSTOM_FITTERS["extratrees"] = fit_extratrees\n_CUSTOM_PARAMS.update(EXTRATREES) '

### 5+c. `nn` slot → `mlp_linear`

Custom stand-in for the **neural** family. Edit only this block.
(`activation="identity"` ≈ linear map through a hidden layer.)


In [61]:
""" # --- collaborator edit: nn / mlp_linear ---
MLP_LINEAR = {
    "MLP_HIDDEN": (32,),
    "MLP_MAX_ITER": 300,
}


def fit_mlp_linear(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    hidden = tuple(MLP_LINEAR["MLP_HIDDEN"])
    model = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation="identity",
        solver="adam",
        max_iter=int(MLP_LINEAR["MLP_MAX_ITER"]),
        random_state=RANDOM_STATE,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={"hidden": str(hidden)},
    )


_CUSTOM_FITTERS["mlp_linear"] = fit_mlp_linear
_CUSTOM_PARAMS.update(MLP_LINEAR) """


' # --- collaborator edit: nn / mlp_linear ---\nMLP_LINEAR = {\n    "MLP_HIDDEN": (32,),\n    "MLP_MAX_ITER": 300,\n}\n\n\ndef fit_mlp_linear(result: PreprocessResult) -> FitBundle:\n    train = result.folds["train"]\n    hidden = tuple(MLP_LINEAR["MLP_HIDDEN"])\n    model = MLPRegressor(\n        hidden_layer_sizes=hidden,\n        activation="identity",\n        solver="adam",\n        max_iter=int(MLP_LINEAR["MLP_MAX_ITER"]),\n        random_state=RANDOM_STATE,\n    )\n    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))\n    return FitBundle(\n        model=model,\n        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),\n        summary_extras={"hidden": str(hidden)},\n    )\n\n\n_CUSTOM_FITTERS["mlp_linear"] = fit_mlp_linear\n_CUSTOM_PARAMS.update(MLP_LINEAR) '

#### Wire-up

Not a lin/tr/nn model block. Errors if `CUSTOM_MODEL_TYPES` names a model that
was not registered by 5+a/b/c; then builds `fit_custom` / `CUSTOM`.


In [62]:
_unknown = [name for name in CUSTOM_MODEL_TYPES if name not in _CUSTOM_FITTERS]
if _unknown:
    raise ValueError(
        "CUSTOM_MODEL_TYPES names are not registered in §5+a/b/c "
        f"(no fit_* block): {_unknown}. "
        f"Registered: {sorted(_CUSTOM_FITTERS)}."
    )
_unused = sorted(set(_CUSTOM_FITTERS) - set(CUSTOM_MODEL_TYPES))
if _unused:
    print("note: registered but not in CUSTOM_MODEL_TYPES:", _unused)


def fit_custom(model_type: str, result: PreprocessResult) -> FitBundle:
    try:
        fitter = _CUSTOM_FITTERS[model_type]
    except KeyError as exc:
        raise ValueError(
            f"Unknown custom model_type={model_type!r}; "
            f"registered={sorted(_CUSTOM_FITTERS)}."
        ) from exc
    return fitter(result)


CUSTOM = {
    "MODEL_TYPES": tuple(CUSTOM_MODEL_TYPES),
    "STANDARDIZE": bool(CUSTOM_STANDARDIZE),
    **_CUSTOM_PARAMS,
}

ACTIVE_FAMILIES = [
    ("custom", dict(CUSTOM)) if name == "custom" else (name, block)
    for name, block in ACTIVE_FAMILIES
]
if ACTIVE_FAMILIES and ACTIVE_FAMILIES[0][0] == "custom":
    STANDARDIZE = bool(CUSTOM["STANDARDIZE"])

print("fit_custom ready:", CUSTOM["MODEL_TYPES"])
print("registered:", sorted(_CUSTOM_FITTERS))


fit_custom ready: ()
registered: []


## 6. Fit And Predict

For each active family: `lin`/`tr`/`nn` via `apply_demo_block`; `custom` via
`build_demo_custom_spec` + §5+ `fit_custom`. Then re-preprocess and `run_one_model`.


In [63]:
from src.metrics import infer_task
from src.pipeline import run_one_model
from src.preprocess import run_preprocess
from src.train import apply_demo_block, build_demo_custom_spec

In [64]:
task = infer_task(TARGET)
if task != "regression":
    raise NotImplementedError(f"Demo supports regression only; got TARGET={TARGET!r}")

summary_rows = []
fit_rows = []
for family, block in ACTIVE_FAMILIES:
    if family == "custom":
        spec = build_demo_custom_spec(
            block,
            fit_one=fit_custom,
            feature_importance=FEATURE_IMPORTANCE,
            feature_importance_repeats=FEATURE_IMPORTANCE_REPEATS,
            random_state=RANDOM_STATE,
        )
    else:
        spec = apply_demo_block(
            family,
            block,
            feature_importance=FEATURE_IMPORTANCE,
            feature_importance_repeats=FEATURE_IMPORTANCE_REPEATS,
            random_state=RANDOM_STATE,
        )
    result_f = run_preprocess(
        df,
        target=TARGET,
        feature_columns=FEATURE_COLUMNS,
        split_column=SPLIT_COLUMN,
        train_end=TRAIN_END,
        val_end=VAL_END,
        impute_strategy=IMPUTE_STRATEGY,
        standardize=bool(block["STANDARDIZE"]),
    )
    train_y_mean_f = float(result_f.folds["train"].y.to_numpy(dtype=float).mean())
    print(f"\n### family={family}  standardize={block['STANDARDIZE']}  "
          f"models={list(spec.model_types)}")
    for model_type in spec.model_types:
        row, _predictions, record = run_one_model(
            spec,
            model_type=model_type,
            result=result_f,
            train_y_mean=train_y_mean_f,
            task=task,
        )
        summary_rows.append(row)
        fit_rows.append({"family": family, **record})

summary = pd.DataFrame(summary_rows)
fits = pd.DataFrame(fit_rows)


### family=lin  standardize=True  models=['ols', 'ridge', 'lasso']

OLS

------------------------------------------------------------
METRICS
------------------------------------------------------------
           n         r2 (oos)        rmse (mean/zero)         mae (mean/zero)      ic rank_ic hit_rate
fold                                                                                                  
train  10736  0.0030 (0.0030)  0.0711 (0.0713/0.0713)  0.0497 (0.0496/0.0497)  0.0551  0.0415   0.5142
val     2861  0.0058 (0.0063)  0.0748 (0.0750/0.0751)  0.0531 (0.0532/0.0532)  0.0915  0.0831   0.5163
test    5883  0.0060 (0.0060)  0.0841 (0.0843/0.0844)  0.0596 (0.0598/0.0597)  0.0958  0.0684   0.5156

------------------------------------------------------------
QUINTILES
------------------------------------------------------------
fold    n      q1      q2      q3     q4     q5  top_minus_bottom
 val 2861 -0.0005 -0.0021  0.0034 0.0038 0.0200            0.0205
test 5883 -0.004

## 7. Summary

Compact cross-model comparison on the test fold (`print_summary_table`).


In [ ]:
print(summary)
# print(fits)

   model  n_test  oos_r2      r2  quintile_ls  rank_ic     ic    mae   rmse  hit_rate
0    ols    5883  0.0060  0.0060       0.0324   0.0684 0.0958 0.0596 0.0841    0.5156
1  ridge    5883  0.0060  0.0060       0.0324   0.0685 0.0958 0.0596 0.0841    0.5156
2  lasso    5883  0.0000 -0.0000          NaN      NaN    NaN 0.0598 0.0843    0.4965
  family  model hidden_sizes  selected_alpha selected_max_depth best_iteration  random_state  val_oos_r2  val_rmse
0    lin    ols         None             NaN               None           None            42      0.0063    0.0748
1    lin  ridge         None          0.0100               None           None            42      0.0063    0.0748
2    lin  lasso         None          0.0100               None           None            42      0.0000    0.0750
